# Frame-level Conditional LayoutDM (8_layout_train)

- 10fps frame 단위 decoding
- bbox = 4 discrete tokens (cx, cy, w, h), 각 80 bin + 1 MASK
- D3PM absorbing state diffusion (MaskGIT-style sampling)
- Conditioning: channel/video/STT 임베딩 + prev-frame active layout
- Channel zone codebook은 cross-attn KV로 inject (centroid prior PE)
- 추론: T=20 step iterative unmask
- Logit adjustment: role prior 만 (off-screen/overlap은 sample-level rerank)
- 평가: selected IoU, best-of-N IoU, NLL


In [1]:
# %% 셀 1: Setup
import os, json, math, random, unicodedata
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import polars as pl
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
from sklearn.cluster import KMeans

POS_DIR = "./data/8_telop_position"
STT_DIR = "./data/4_stt_results"
EMB_PATH = "./data/8_text_embeddings.pt"
SAVE_DIR = "./model/8_layout_train"
os.makedirs(SAVE_DIR, exist_ok=True)

# ── 절대 변하지 않는 상수 ──
GRID_W = 80
GRID_H = 80
FPS = 10
MAX_FRAMES = 2000
MAX_TEXT_LEN = 200
MAX_ACTIVE_PER_FRAME = 150

# ── Tokenization ──
N_BINS = 80         # cx/cy/w/h 모두 80 bin (720x1280을 이미 80x80으로 압축한 상태)
MASK_ID = N_BINS    # 80 = [MASK]
VOCAB = N_BINS + 1  # 81

# ── 학습 데이터 ──
SEED = 42
NUM_WORKERS = 32
EVAL_PER_CHANNEL = 5

# ── 모델 hyperparam ──
D_MODEL = 256
N_HEADS = 8
N_LAYERS = 6
N_LAYERS_CROSS = 2
DROPOUT = 0.1
D_AXIS = 16
N_CTX_GLOBAL_MAX = 3       # CH + VID + (0 or 1) STT

# ── Channel zone codebook ──
K_ZONES = 16

# ── 학습 ──
BATCH_SIZE = 512
NUM_WORKERS_DL = 32
PREFETCH_FACTOR = 4
EPOCHS = 15
LR = 2e-4
WEIGHT_DECAY = 1e-2
GRAD_CLIP = 1.0
# prev_bbox 4개 feature는 제거 (instance 대표 bbox라 GT-cheat 위험). prev_exists / lifecycle만 남김.
SUBSAMPLE_CHANNELS = 67

# ── Set-level loss ──
LAMBDA_OVERLAP = 1.0       # differentiable pairwise IoU loss (training)
TEMP_OVERLAP = 0.25        # softmax temperature for expected bbox (sharper → 강한 gradient)

# ── Sampling ──
T_INFER = 20
N_RERANK_SAMPLES = 8
ROLE_PRIOR_LAMBDA = 0.0    # 초기 baseline: centroid prior 끄기 (이전 template/centroid 실패 교훈)
ROLE_PRIOR_SIGMA = 0.05    # K-means centroid 주변 Gaussian sigma (normalized [0,1])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={DEVICE}  VOCAB={VOCAB} (mask={MASK_ID})")

text2emb = torch.load(EMB_PATH, weights_only=True)
EMB_DIM = next(iter(text2emb.values())).shape[0]
ZERO_EMB = torch.zeros(EMB_DIM)
print(f"✅ 임베딩 로드: {len(text2emb):,}개  |  dim={EMB_DIM}")


device=cuda  VOCAB=81 (mask=80)
✅ 임베딩 로드: 2,167,019개  |  dim=1024


In [2]:
# %% 셀 2: 토크나이저
def bbox_to_tokens(gx, gy, gw, gh):
    """grid_x in [0, GRID_W), grid_y in [0, GRID_H), grid_w/h in [1, GRID_W] → 4 bins in [0, N_BINS)."""
    cx = int(np.clip(round(gx), 0, N_BINS - 1))
    cy = int(np.clip(round(gy), 0, N_BINS - 1))
    w  = int(np.clip(round(gw) - 1, 0, N_BINS - 1))    # gw ∈ [1, 80] → bin ∈ [0, 79]
    h  = int(np.clip(round(gh) - 1, 0, N_BINS - 1))
    return cx, cy, w, h


def tokens_to_bbox(cx_bin, cy_bin, w_bin, h_bin):
    """reverse: returns (cx_norm, cy_norm, w_norm, h_norm) in [0, 1]."""
    cx = (cx_bin + 0.5) / N_BINS
    cy = (cy_bin + 0.5) / N_BINS
    w  = (w_bin + 1) / N_BINS
    h  = (h_bin + 1) / N_BINS
    return cx, cy, w, h


In [3]:
# %% 셀 3: 데이터 로드 + train/eval split + 채널 서브샘플링
def stt_frames_to_segments(df_stt):
    rows = df_stt.sort("frame_num").to_dicts()
    segments, cur_text, cur_start_frame, prev_frame = [], None, None, None
    for r in rows:
        t, f = r["stt_text"], int(r["frame_num"])
        if t != cur_text:
            if cur_text is not None and cur_text != "":
                segments.append({"start": cur_start_frame / FPS,
                                 "end": (prev_frame + 1) / FPS,
                                 "text": cur_text.strip()})
            cur_text, cur_start_frame = t, f
        prev_frame = f
    if cur_text is not None and cur_text != "":
        segments.append({"start": cur_start_frame / FPS,
                         "end": (prev_frame + 1) / FPS,
                         "text": cur_text.strip()})
    return segments


def text_compose(text):
    """한글/영문/숫자/공백/기타 count + 전각 비율. width 예측에 도움."""
    n_ko = n_en = n_num = n_sp = n_etc = 0
    n_fw = 0
    n_total = len(text)
    for c in text:
        if '가' <= c <= '힣':
            n_ko += 1
        elif c.isascii() and c.isalpha():
            n_en += 1
        elif c.isdigit():
            n_num += 1
        elif c.isspace():
            n_sp += 1
        else:
            n_etc += 1
        if unicodedata.east_asian_width(c) in ('W', 'F'):
            n_fw += 1
    fw_ratio = n_fw / max(n_total, 1)
    return n_ko, n_en, n_num, n_sp, n_etc, fw_ratio


def load_one(args):
    """SoA (struct-of-arrays) 구조로 영상별 데이터 로드 — 벡터화 친화."""
    channel, path = args
    with open(path, "r") as f:
        data = json.load(f)
    instances = data.get("instances", [])
    duration = data.get("duration", 0.1)
    if instances:
        duration = max(duration, max(inst["end_sec"] for inst in instances))
    video_name = data.get("video", "")
    file_id = os.path.basename(path)[:-5]

    inst_texts, inst_starts, inst_ends = [], [], []
    inst_start_fs, inst_end_fs, inst_tokens_list, inst_text_lens = [], [], [], []
    inst_compose = []   # (n_ko, n_en, n_num, n_sp, n_etc, fw_ratio) per instance
    for inst in instances:
        cx_bin, cy_bin, w_bin, h_bin = bbox_to_tokens(inst["grid_x"], inst["grid_y"],
                                                       inst["grid_w"], inst["grid_h"])
        text = inst["text"][:MAX_TEXT_LEN]
        inst_texts.append(text)
        inst_starts.append(inst["start_sec"])
        inst_ends.append(inst["end_sec"])
        inst_start_fs.append(int(inst["start_sec"] * FPS))
        inst_end_fs.append(int(inst["end_sec"] * FPS))
        inst_tokens_list.append([cx_bin, cy_bin, w_bin, h_bin])
        inst_text_lens.append(len(text))
        inst_compose.append(list(text_compose(text)))

    stt_path = os.path.join(STT_DIR, channel, file_id + ".parquet")
    stt_segments = []
    if os.path.exists(stt_path):
        try:
            df_stt = pl.read_parquet(stt_path, glob=False)
            stt_segments = stt_frames_to_segments(df_stt)
        except Exception:
            pass

    n_frames = min(int(duration * FPS) + 1, MAX_FRAMES)
    inst_start_arr = np.array(inst_start_fs, dtype=np.int64) if inst_start_fs else np.zeros(0, dtype=np.int64)
    inst_end_arr   = np.array(inst_end_fs,   dtype=np.int64) if inst_end_fs   else np.zeros(0, dtype=np.int64)

    # ── 활성 frame 인덱스 (정렬된 list) ──
    active_set = set()
    for sf, ef in zip(inst_start_arr.tolist(), inst_end_arr.tolist()):
        for f in range(max(0, sf), min(n_frames, ef + 1)):
            active_set.add(f)
    active_frames = sorted(active_set)

    # ── frame → STT segment 인덱스 (0개 또는 1개 — 미리 계산) ──
    stt_texts = [seg["text"] for seg in stt_segments]
    frame_to_stt = np.full(n_frames, -1, dtype=np.int32)
    for seg_idx, seg in enumerate(stt_segments):
        sf = max(0, int(seg["start"] * FPS))
        ef = min(n_frames, int(seg["end"] * FPS))
        if ef > sf:
            frame_to_stt[sf:ef] = seg_idx

    return {
        "channel": channel,
        "video_name": video_name,
        "file_id": file_id,
        "duration": duration,
        "n_frames": n_frames,
        "active_frames": active_frames,
        # ── Instance SoA ──
        "inst_texts":    inst_texts,
        "inst_start":    np.array(inst_starts, dtype=np.float32) if inst_starts else np.zeros(0, dtype=np.float32),
        "inst_end":      np.array(inst_ends,   dtype=np.float32) if inst_ends   else np.zeros(0, dtype=np.float32),
        "inst_start_f":  inst_start_arr,
        "inst_end_f":    inst_end_arr,
        "inst_tokens":   np.array(inst_tokens_list, dtype=np.int64) if inst_tokens_list else np.zeros((0, 4), dtype=np.int64),
        "inst_text_len": np.array(inst_text_lens,   dtype=np.int64) if inst_text_lens   else np.zeros(0, dtype=np.int64),
        "inst_compose":  np.array(inst_compose,     dtype=np.float32) if inst_compose else np.zeros((0, 6), dtype=np.float32),
        # ── STT ──
        "stt_texts": stt_texts,
        "frame_to_stt": frame_to_stt,
    }


paths_by_channel = defaultdict(list)
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for fname in os.listdir(ch_dir):
        if fname.endswith(".json"):
            paths_by_channel[channel].append(os.path.join(ch_dir, fname))

channels_all = sorted(paths_by_channel.keys())
print(f"📁 전체 채널: {len(channels_all):,}")

rng = random.Random(SEED)
train_paths, eval_paths = [], []
for ch in channels_all:
    paths = sorted(paths_by_channel[ch])
    rng.shuffle(paths)
    eval_paths.extend([(ch, p) for p in paths[:EVAL_PER_CHANNEL]])
    train_paths.extend([(ch, p) for p in paths[EVAL_PER_CHANNEL:]])

print(f"📁 train paths={len(train_paths):,}  eval paths={len(eval_paths):,}")


def load_parallel(args_list, desc):
    out = []
    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
        futures = [pool.submit(load_one, a) for a in args_list]
        for fut in tqdm(as_completed(futures), total=len(futures), desc=desc):
            out.append(fut.result())
    return out


train_samples = load_parallel(train_paths, "load-train")
eval_samples = load_parallel(eval_paths, "load-eval")

# 채널 서브샘플링 (debug, 이전 컨벤션 유지)
rng_ch = random.Random(SEED)
sampled_channels = rng_ch.sample(channels_all, SUBSAMPLE_CHANNELS)
sampled_set = set(sampled_channels)

train_samples = [s for s in train_samples if s["channel"] in sampled_set and len(s["active_frames"]) > 0 and len(s["inst_tokens"]) > 0]
eval_samples = [s for s in eval_samples if s["channel"] in sampled_set and len(s["active_frames"]) > 0 and len(s["inst_tokens"]) > 0]
channels = sampled_channels
channel2id = {ch: i for i, ch in enumerate(channels)}
N_CHANNELS = len(channels)

print(f"\n🔬 서브샘플링: {N_CHANNELS}개 채널")
print(f"   train={len(train_samples):,}  eval={len(eval_samples):,}")


📁 전체 채널: 664
📁 train paths=63,080  eval paths=3,320


load-eval: 100%|██████████| 3320/3320 [00:01<00:00, 2013.76it/s]



🔬 서브샘플링: 67개 채널
   train=5,574  eval=288


In [4]:
# %% 셀 4: 채널별 K-means centroid (role prior + cross-attn PE)
print("📊 채널별 K-means centroid (normalized xy)")
channel_centroids = np.full((N_CHANNELS, K_ZONES, 2), 0.5, dtype=np.float32)
channel_weights = np.full((N_CHANNELS, K_ZONES), 1.0 / K_ZONES, dtype=np.float32)

ch2xy = defaultdict(list)
for s in train_samples:
    cid = channel2id[s["channel"]]
    if len(s["inst_tokens"]) == 0:
        continue
    xy_norm = (s["inst_tokens"][:, :2].astype(np.float32) + 0.5) / N_BINS
    ch2xy[cid].append(xy_norm)

for cid in tqdm(range(N_CHANNELS), desc="K-means"):
    arrs = ch2xy.get(cid, [])
    if not arrs:
        continue
    pts = np.concatenate(arrs, axis=0)
    if len(pts) == 0:
        continue
    k_eff = min(K_ZONES, max(1, len(pts)))
    km = KMeans(n_clusters=k_eff, random_state=SEED, n_init=5).fit(pts)
    cents = km.cluster_centers_.astype(np.float32)
    labels = km.labels_
    weights = np.zeros(K_ZONES, dtype=np.float32)
    for k in range(k_eff):
        weights[k] = (labels == k).sum() / len(labels)
    if k_eff < K_ZONES:
        pad = np.full((K_ZONES - k_eff, 2), 0.5, dtype=np.float32)
        cents = np.concatenate([cents, pad], axis=0)
    channel_centroids[cid] = cents
    channel_weights[cid] = weights

channel_centroids_t = torch.from_numpy(channel_centroids)        # (C, K, 2)
channel_weights_t = torch.from_numpy(channel_weights)            # (C, K)
print(f"✅ centroid {channel_centroids_t.shape}, weights {channel_weights_t.shape}")

# 미리 계산: 각 채널의 (cx_bin, cy_bin)에 대한 log role prior
print("📊 logit adjustment용 role prior log-prob 테이블")
bin_centers = (torch.arange(N_BINS, dtype=torch.float32) + 0.5) / N_BINS   # (N_BINS,)
role_log_p_x = torch.zeros(N_CHANNELS, N_BINS)
role_log_p_y = torch.zeros(N_CHANNELS, N_BINS)
sig2 = ROLE_PRIOR_SIGMA ** 2
for cid in range(N_CHANNELS):
    cents = channel_centroids_t[cid]    # (K, 2)
    w = channel_weights_t[cid]
    # p(x) = Σ_k w_k N(x | mu_kx, sigma)  marginal
    diff_x = bin_centers.unsqueeze(0) - cents[:, 0:1]                # (K, N_BINS)
    log_n_x = -0.5 * (diff_x ** 2) / sig2 - 0.5 * math.log(2 * math.pi * sig2)
    log_p_x = torch.logsumexp(log_n_x + torch.log(w + 1e-9).unsqueeze(-1), dim=0)
    diff_y = bin_centers.unsqueeze(0) - cents[:, 1:2]
    log_n_y = -0.5 * (diff_y ** 2) / sig2 - 0.5 * math.log(2 * math.pi * sig2)
    log_p_y = torch.logsumexp(log_n_y + torch.log(w + 1e-9).unsqueeze(-1), dim=0)
    role_log_p_x[cid] = log_p_x - log_p_x.logsumexp(0)               # normalize
    role_log_p_y[cid] = log_p_y - log_p_y.logsumexp(0)
print(f"✅ role_log_p_x {role_log_p_x.shape}, role_log_p_y {role_log_p_y.shape}")


📊 채널별 K-means centroid (normalized xy)


K-means: 100%|██████████| 67/67 [00:01<00:00, 34.30it/s]

✅ centroid torch.Size([67, 16, 2]), weights torch.Size([67, 16])
📊 logit adjustment용 role prior log-prob 테이블
✅ role_log_p_x torch.Size([67, 80]), role_log_p_y torch.Size([67, 80])


In [5]:
# %% 셀 5: Dataset (frame-level)
from torch.utils.data import Dataset, DataLoader

# 학습 데이터셋 인덱싱: (video_idx, sampled_frame_idx_in_video)
# epoch마다 frame을 새로 sample하므로 Dataset은 (video, frame) 쌍을 동적 생성
class FrameLayoutDataset(Dataset):
    def __init__(self, samples, channel2id, text2emb):
        self.samples = samples
        self.channel2id = channel2id
        self.text2emb = text2emb
        # 모든 (video, active_frame) 쌍 enumerate
        self._index = []
        for vi, s in enumerate(samples):
            for f in s["active_frames"]:
                self._index.append((vi, f))

    def __len__(self):
        return len(self._index)

    def _normalize_text(self, text):
        return F.normalize(self.text2emb.get(text, ZERO_EMB).unsqueeze(0), p=2, dim=1).squeeze(0)

    def __getitem__(self, idx):
        vi, f = self._index[idx]
        s = self.samples[vi]
        cid = self.channel2id[s["channel"]]

        # ── (벡터화) 활성 instance 인덱스 ──
        start_f_all = s["inst_start_f"]
        end_f_all = s["inst_end_f"]
        active_mask = (start_f_all <= f) & (f <= end_f_all)
        active = np.flatnonzero(active_mask)[:MAX_ACTIVE_PER_FRAME]
        n_cur = int(active.shape[0])

        # ── 글로벌 condition ──
        ch_emb = self._normalize_text(s["channel"])
        vid_emb = self._normalize_text(s["video_name"])
        stt_idx = int(s["frame_to_stt"][f])
        stt_emb_f = self._normalize_text(s["stt_texts"][stt_idx]) if stt_idx >= 0 else None

        global_emb_list = [ch_emb, vid_emb]
        global_type_list = [0, 1]
        if stt_emb_f is not None:
            global_emb_list.append(stt_emb_f)
            global_type_list.append(2)
        global_emb = torch.stack(global_emb_list)
        global_type = torch.tensor(global_type_list, dtype=torch.long)
        n_global = global_emb.shape[0]

        # ── instance text embeddings (text2emb dict lookup만 Python, 나머지 벡터) ──
        active_list = active.tolist()
        cur_text_embs = torch.stack([
            self.text2emb.get(s["inst_texts"][i], ZERO_EMB) for i in active_list
        ])
        cur_text_embs = F.normalize(cur_text_embs, p=2, dim=1)

        cur_diff_ch = cur_text_embs - ch_emb.unsqueeze(0)
        cur_diff_vid = cur_text_embs - vid_emb.unsqueeze(0)
        if stt_emb_f is not None:
            cur_diff_stt = cur_text_embs - stt_emb_f.unsqueeze(0)
            cur_stt_sim = F.cosine_similarity(cur_text_embs, stt_emb_f.unsqueeze(0), dim=-1)
            cur_has_stt = torch.ones(n_cur)
        else:
            cur_diff_stt = torch.zeros(n_cur, EMB_DIM)
            cur_stt_sim = torch.zeros(n_cur)
            cur_has_stt = torch.zeros(n_cur)

        ch_sim = F.cosine_similarity(cur_text_embs, ch_emb.unsqueeze(0), dim=-1)
        vid_sim = F.cosine_similarity(cur_text_embs, vid_emb.unsqueeze(0), dim=-1)

        # ── (벡터화) scalar features ──
        active_starts   = s["inst_start"][active]
        active_ends     = s["inst_end"][active]
        active_start_fs = start_f_all[active]
        active_end_fs   = end_f_all[active]
        active_textlen  = s["inst_text_len"][active]
        active_compose  = s["inst_compose"][active]   # (n_cur, 6)
        duration = max(s["duration"], 0.1)

        log_text_len = np.log1p(np.minimum(active_textlen, MAX_TEXT_LEN)).astype(np.float32)
        start_norm   = (active_starts / duration).astype(np.float32)
        end_norm     = (active_ends / duration).astype(np.float32)
        dur_norm     = ((active_ends - active_starts) / duration).astype(np.float32)
        f_prev = f - 1
        prev_exists  = ((active_start_fs <= f_prev) & (f_prev <= active_end_fs) & (f_prev >= 0)).astype(np.float32)
        is_entering  = (active_start_fs == f).astype(np.float32)
        is_exit_3    = ((active_end_fs - f) <= 3).astype(np.float32)
        lifespan     = np.maximum(active_end_fs - active_start_fs, 1).astype(np.float32)
        age_norm     = ((f - active_start_fs).astype(np.float32) / lifespan).astype(np.float32)
        remain_norm  = ((active_end_fs - f).astype(np.float32) / lifespan).astype(np.float32)

        cur_scalars = torch.zeros(n_cur, 19)
        cur_scalars[:, 0]  = torch.from_numpy(log_text_len)
        cur_scalars[:, 1]  = ch_sim
        cur_scalars[:, 2]  = vid_sim
        cur_scalars[:, 3]  = cur_stt_sim
        cur_scalars[:, 4]  = cur_has_stt
        cur_scalars[:, 5]  = torch.from_numpy(start_norm)
        cur_scalars[:, 6]  = torch.from_numpy(end_norm)
        cur_scalars[:, 7]  = torch.from_numpy(dur_norm)
        cur_scalars[:, 8]  = torch.from_numpy(prev_exists)
        cur_scalars[:, 9]  = torch.from_numpy(is_entering)
        cur_scalars[:, 10] = torch.from_numpy(is_exit_3)
        cur_scalars[:, 11] = torch.from_numpy(age_norm)
        cur_scalars[:, 12] = torch.from_numpy(remain_norm)
        # text composition (n_ko, n_en, n_num, n_sp, n_etc log1p + fw_ratio)
        cur_scalars[:, 13] = torch.from_numpy(np.log1p(active_compose[:, 0]))
        cur_scalars[:, 14] = torch.from_numpy(np.log1p(active_compose[:, 1]))
        cur_scalars[:, 15] = torch.from_numpy(np.log1p(active_compose[:, 2]))
        cur_scalars[:, 16] = torch.from_numpy(np.log1p(active_compose[:, 3]))
        cur_scalars[:, 17] = torch.from_numpy(np.log1p(active_compose[:, 4]))
        cur_scalars[:, 18] = torch.from_numpy(active_compose[:, 5])

        # ── (벡터화) GT bbox tokens ──
        cur_gt_tokens = torch.from_numpy(s["inst_tokens"][active]).long()
        # 동시 텔롭 판별용 시간 구간 (set-level loss에서 사용)
        gt_times = torch.from_numpy(np.stack([active_starts, active_ends], axis=1)).float()
        frame_pos = torch.tensor([f / max(s["n_frames"], 1)], dtype=torch.float32)

        return {
            "channel_id": cid,
            "global_emb": global_emb,
            "global_type": global_type,
            "cur_text": cur_text_embs,
            "cur_diff_ch": cur_diff_ch,
            "cur_diff_vid": cur_diff_vid,
            "cur_diff_stt": cur_diff_stt,
            "cur_scalars": cur_scalars,
            "cur_gt_tokens": cur_gt_tokens,
            "gt_times": gt_times,
            "frame_pos": frame_pos,
            "n_global": n_global,
            "n_cur": n_cur,
        }


def collate(batch):
    G_MAX = max(b["n_global"] for b in batch)
    C_MAX = max(b["n_cur"] for b in batch)

    def pad2d(t, max_n, pad_val=0.0, dtype=None):
        n = t.shape[0]
        if dtype is None: dtype = t.dtype
        rest = t.shape[1:] if t.ndim > 1 else ()
        out = torch.full((max_n, *rest), pad_val, dtype=dtype)
        if n > 0:
            out[:n] = t
        return out

    def pad1d(t, max_n, pad_val=0, dtype=None):
        n = t.shape[0]
        if dtype is None: dtype = t.dtype
        out = torch.full((max_n,), pad_val, dtype=dtype)
        if n > 0:
            out[:n] = t
        return out

    out = {
        "channel_id": torch.tensor([b["channel_id"] for b in batch], dtype=torch.long),
        "frame_pos": torch.stack([b["frame_pos"] for b in batch]),
        "global_emb":  torch.stack([pad2d(b["global_emb"], G_MAX) for b in batch]),
        "global_type": torch.stack([pad1d(b["global_type"], G_MAX, 0) for b in batch]),
        "global_mask": torch.stack([
            torch.cat([torch.ones(b["n_global"]), torch.zeros(G_MAX - b["n_global"])])
            for b in batch
        ]).bool(),
        "cur_text":     torch.stack([pad2d(b["cur_text"], C_MAX) for b in batch]),
        "cur_diff_ch":  torch.stack([pad2d(b["cur_diff_ch"], C_MAX) for b in batch]),
        "cur_diff_vid": torch.stack([pad2d(b["cur_diff_vid"], C_MAX) for b in batch]),
        "cur_diff_stt": torch.stack([pad2d(b["cur_diff_stt"], C_MAX) for b in batch]),
        "cur_scalars":  torch.stack([pad2d(b["cur_scalars"], C_MAX) for b in batch]),
        "cur_gt_tokens": torch.stack([pad2d(b["cur_gt_tokens"], C_MAX, dtype=torch.long) for b in batch]),
        "gt_times":     torch.stack([pad2d(b["gt_times"], C_MAX) for b in batch]),
        "cur_mask": torch.stack([
            torch.cat([torch.ones(b["n_cur"]), torch.zeros(C_MAX - b["n_cur"])])
            for b in batch
        ]).bool(),
    }
    return out


train_ds = FrameLayoutDataset(train_samples, channel2id, text2emb)
eval_ds = FrameLayoutDataset(eval_samples, channel2id, text2emb)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=NUM_WORKERS_DL, collate_fn=collate, drop_last=True,
                      pin_memory=True, persistent_workers=True, prefetch_factor=PREFETCH_FACTOR)
eval_dl = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False,
                     num_workers=NUM_WORKERS_DL, collate_fn=collate,
                     pin_memory=True, persistent_workers=True, prefetch_factor=PREFETCH_FACTOR)
print(f"📊 train batches={len(train_dl)}, eval batches={len(eval_dl)}")


📊 train batches=3971, eval batches=204


In [6]:
# %% 셀 6: 모델 (LayoutDM Transformer)
def sinusoidal_pe(x, d, scale=50.0):
    """x: (..., 1) in any range. Returns sinusoidal PE of dim d (d % 2 == 0)."""
    assert d % 2 == 0
    div = torch.exp(-torch.arange(0, d // 2, device=x.device, dtype=x.dtype)
                     * (math.log(10000.0) / (d // 2)))
    a = x * scale * div
    return torch.cat([torch.sin(a), torch.cos(a)], dim=-1)


def pe_2d(xy, d, scale=50.0):
    assert d % 4 == 0
    div = torch.exp(-torch.arange(0, d // 4, device=xy.device, dtype=xy.dtype)
                     * (math.log(10000.0) / (d // 4)))
    x = xy[..., 0:1] * scale * div
    y = xy[..., 1:2] * scale * div
    return torch.cat([torch.sin(x), torch.cos(x), torch.sin(y), torch.cos(y)], dim=-1)


class LayoutDM(nn.Module):
    def __init__(self, n_channels, emb_dim=EMB_DIM, d_model=D_MODEL, K=K_ZONES,
                 n_heads=N_HEADS, n_layers=N_LAYERS, n_layers_cross=N_LAYERS_CROSS,
                 dropout=DROPOUT, centroid_init=None):
        super().__init__()
        self.K = K
        self.d_model = d_model

        # ── continuous condition projection ──
        # global: channel/video/stt emb (1024d)
        self.global_proj = nn.Sequential(
            nn.Linear(emb_dim, d_model), nn.GELU(),
            nn.Linear(d_model, d_model))
        self.global_type_emb = nn.Embedding(3, d_model)   # ch/vid/stt

        # cur ctx: text + 3 diff + 19 scalars (lifecycle + stt_sim + text composition 포함)
        self.cur_text_proj  = nn.Sequential(nn.Linear(emb_dim, 64), nn.GELU())
        self.cur_dch_proj   = nn.Sequential(nn.Linear(emb_dim, 64), nn.GELU())
        self.cur_dvid_proj  = nn.Sequential(nn.Linear(emb_dim, 64), nn.GELU())
        self.cur_dstt_proj  = nn.Sequential(nn.Linear(emb_dim, 64), nn.GELU())
        self.cur_scalar_proj = nn.Sequential(nn.Linear(19, 64), nn.GELU())
        self.cur_ctx_combine = nn.Sequential(
            nn.Linear(64 * 5, d_model), nn.GELU(),
            nn.Linear(d_model, d_model))

        # ── discrete bbox token embedding (shared 0~N_BINS + MASK) ──
        self.bbox_emb = nn.Embedding(VOCAB, d_model)
        # axis-id embedding (4 axes)
        self.axis_emb = nn.Embedding(4, d_model)
        # token type embedding (0=global, 1=cur_ctx, 2=cur_bbox)
        self.type_emb = nn.Embedding(3, d_model)
        # instance-id embedding (for grouping bbox tokens of same instance)
        self.inst_id_emb = nn.Embedding(MAX_ACTIVE_PER_FRAME, d_model)
        # frame position
        self.frame_proj = nn.Linear(d_model, d_model)
        # channel bias
        self.channel_bias = nn.Embedding(n_channels, d_model)

        # ── Self-attention encoder ──
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
                                            dim_feedforward=d_model * 4, dropout=dropout,
                                            batch_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers,
                                              enable_nested_tensor=False)

        # ── Channel zone codebook + cross-attn (after encoder) ──
        self.zone_codebook = nn.Embedding(n_channels * K, d_model // 2)
        nn.init.normal_(self.zone_codebook.weight, std=0.02)
        if centroid_init is not None:
            self.register_buffer("centroid_init", centroid_init)
        else:
            self.register_buffer("centroid_init", torch.full((n_channels, K, 2), 0.5))
        self.zone_proj = nn.Linear(d_model // 2 + d_model // 2, d_model)

        self.cross_attn = nn.ModuleList([
            nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
            for _ in range(n_layers_cross)])
        self.cross_norm1 = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers_cross)])
        self.cross_ff = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_model * 4), nn.GELU(),
                          nn.Dropout(dropout), nn.Linear(d_model * 4, d_model))
            for _ in range(n_layers_cross)])
        self.cross_norm2 = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers_cross)])

        # ── Output heads: 4 axis-specific, each predicting N_BINS logits ──
        self.head_cx = nn.Linear(d_model, N_BINS)
        self.head_cy = nn.Linear(d_model, N_BINS)
        self.head_w  = nn.Linear(d_model, N_BINS)
        self.head_h  = nn.Linear(d_model, N_BINS)

    def build_sequence(self, batch, cur_bbox_tokens):
        """
        Returns:
          seq:      (B, L, d_model)
          pad_mask: (B, L) — True at PAD
          bbox_pos: (B, n_cur, 4) — index into L for each (i, axis) bbox token
          cur_mask: (B, n_cur) — True at real instance
        """
        B = batch["channel_id"].shape[0]
        G = batch["global_emb"].shape[1]
        C = batch["cur_text"].shape[1]
        device = batch["channel_id"].device

        # ── Global tokens ──
        global_tok = self.global_proj(batch["global_emb"]) + self.global_type_emb(batch["global_type"])
        global_tok = global_tok + self.type_emb.weight[0].unsqueeze(0).unsqueeze(0)

        # ── Cur CTX tokens (1 per active instance — self-prev features는 cur_scalars 안에 포함) ──
        cur_ctx_in = torch.cat([
            self.cur_text_proj(batch["cur_text"]),
            self.cur_dch_proj(batch["cur_diff_ch"]),
            self.cur_dvid_proj(batch["cur_diff_vid"]),
            self.cur_dstt_proj(batch["cur_diff_stt"]),
            self.cur_scalar_proj(batch["cur_scalars"]),
        ], dim=-1)
        cur_ctx_tok = self.cur_ctx_combine(cur_ctx_in)   # (B, C, d)
        inst_ids = torch.arange(C, device=device).unsqueeze(0).expand(B, -1)
        cur_ctx_tok = cur_ctx_tok + self.inst_id_emb(inst_ids)
        cur_ctx_tok = cur_ctx_tok + self.type_emb.weight[1].unsqueeze(0).unsqueeze(0)

        # ── Cur bbox tokens (4 per instance: cx/cy/w/h) ──
        bbox_emb = self.bbox_emb(cur_bbox_tokens)        # (B, C, 4, d)
        axis_ids = torch.arange(4, device=device).unsqueeze(0).unsqueeze(0).expand(B, C, -1)
        bbox_emb = bbox_emb + self.axis_emb(axis_ids)
        bbox_emb = bbox_emb + self.inst_id_emb(inst_ids).unsqueeze(2)
        bbox_emb = bbox_emb + self.type_emb.weight[2].view(1, 1, 1, -1)
        bbox_seq = bbox_emb.reshape(B, C * 4, -1)

        # ── Concat ──
        seq = torch.cat([global_tok, cur_ctx_tok, bbox_seq], dim=1)   # (B, L, d)

        # ── Add channel bias + frame pos to all tokens ──
        ch_bias = self.channel_bias(batch["channel_id"]).unsqueeze(1)
        fp_pe = sinusoidal_pe(batch["frame_pos"], self.d_model)
        fp = self.frame_proj(fp_pe).unsqueeze(1)
        seq = seq + ch_bias + fp

        # ── Padding mask ──
        cur_mask = batch["cur_mask"]                      # (B, C)
        bbox_mask = cur_mask.unsqueeze(-1).expand(-1, -1, 4).reshape(B, C * 4)
        pad_mask = ~torch.cat([batch["global_mask"], cur_mask, bbox_mask], dim=1)

        # ── Bbox positions in sequence ──
        L_pre = G + C
        bbox_pos = torch.arange(C * 4, device=device).view(1, C, 4).expand(B, -1, -1) + L_pre
        return seq, pad_mask, bbox_pos, cur_mask

    def forward(self, batch, cur_bbox_tokens):
        seq, pad_mask, bbox_pos, cur_mask = self.build_sequence(batch, cur_bbox_tokens)
        h = self.encoder(seq, src_key_padding_mask=pad_mask)

        # Cross-attn to channel zone codebook
        B = batch["channel_id"].shape[0]
        idx = batch["channel_id"].unsqueeze(-1) * self.K + \
              torch.arange(self.K, device=batch["channel_id"].device).unsqueeze(0)
        z = self.zone_codebook(idx)                                     # (B, K, d/2)
        pe = pe_2d(self.centroid_init[batch["channel_id"]], self.d_model // 2).to(z.dtype)
        zone_kv = self.zone_proj(torch.cat([z, pe], dim=-1))            # (B, K, d)
        for attn, n1, ff, n2 in zip(self.cross_attn, self.cross_norm1,
                                      self.cross_ff, self.cross_norm2):
            a, _ = attn(h, zone_kv, zone_kv)
            h = n1(h + a)
            h = n2(h + ff(h))

        # Gather bbox positions
        B, C, _ = bbox_pos.shape
        h_flat = h.gather(1, bbox_pos.reshape(B, C * 4, 1).expand(-1, -1, h.shape[-1]))
        h_bbox = h_flat.view(B, C, 4, -1)

        logits_cx = self.head_cx(h_bbox[:, :, 0, :])   # (B, C, N_BINS)
        logits_cy = self.head_cy(h_bbox[:, :, 1, :])
        logits_w  = self.head_w(h_bbox[:, :, 2, :])
        logits_h  = self.head_h(h_bbox[:, :, 3, :])
        return {"cx": logits_cx, "cy": logits_cy, "w": logits_w, "h": logits_h,
                "cur_mask": cur_mask}


torch.manual_seed(SEED)
model = LayoutDM(n_channels=N_CHANNELS, centroid_init=channel_centroids_t).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ 모델: {n_params/1e6:.2f}M params")

# torch.compile은 dynamic shape (가변 C_MAX) 처리 못 해서 사용 안 함


✅ 모델: 7.49M params


In [7]:
# %% 셀 7: Loss + Training step (D3PM CE on masked positions)
def corrupt_tokens(gt_tokens, cur_mask):
    """
    gt_tokens: (B, C, 4) long
    cur_mask: (B, C) bool — True at real instance
    Returns:
      noisy_tokens: (B, C, 4) — masked positions = MASK_ID
      mask_target: (B, C, 4) bool — True at positions to predict (masked + valid)
    """
    B, C, _ = gt_tokens.shape
    device = gt_tokens.device
    # Per-sample r with cosine schedule (LayoutDM/MaskGIT 표준)
    # uniform U(0,1) 대신 cos schedule → 추론 step 분포와 일치
    u = torch.rand(B, 1, 1, device=device)
    r = torch.cos(u * math.pi / 2)
    # Bernoulli(r) per token
    mask = (torch.rand(B, C, 4, device=device) < r) & cur_mask.unsqueeze(-1)
    # 최소 1개는 마스킹 보장 (학습 신호)
    needs_at_least_one = cur_mask.any(-1) & (~mask.any((-1, -2)))
    if needs_at_least_one.any():
        for b in needs_at_least_one.nonzero(as_tuple=False).squeeze(-1).tolist():
            valid_i = cur_mask[b].nonzero(as_tuple=False).squeeze(-1)
            if len(valid_i) > 0:
                i = valid_i[torch.randint(0, len(valid_i), (1,))].item()
                ax = torch.randint(0, 4, (1,)).item()
                mask[b, i, ax] = True

    noisy = gt_tokens.clone()
    noisy[mask] = MASK_ID
    return noisy, mask


def differentiable_overlap_loss(out, gt_tokens, mask_target, gt_times, cur_mask):
    """
    Expected bbox (softmax weighted) 기반 미분 가능 pairwise IoU loss.
    masked 위치는 softmax expectation, unmasked는 GT 사용 → joint reasoning 학습.
    """
    device = gt_tokens.device
    bin_xy = (torch.arange(N_BINS, device=device).float() + 0.5) / N_BINS
    bin_wh = (torch.arange(N_BINS, device=device).float() + 1) / N_BINS

    boxes = {}
    for ax_idx, (ax, bins) in enumerate(zip(["cx", "cy", "w", "h"],
                                              [bin_xy, bin_xy, bin_wh, bin_wh])):
        # Straight-through Gumbel-softmax: forward는 one-hot (argmax with Gumbel noise),
        # backward는 softmax(tau)의 gradient. → forward exp_val이 실제 sampling과 일치
        onehot = F.gumbel_softmax(out[ax].float(), tau=TEMP_OVERLAP, hard=True, dim=-1)
        exp_val = (onehot * bins).sum(-1)
        gt_val = bins[gt_tokens[..., ax_idx]]
        m = mask_target[..., ax_idx]
        boxes[ax] = torch.where(m, exp_val, gt_val)

    cx, cy, w, h = boxes["cx"], boxes["cy"], boxes["w"], boxes["h"]
    x1, x2 = cx - w / 2, cx + w / 2
    y1, y2 = cy - h / 2, cy + h / 2

    B, C = cur_mask.shape
    ix1 = torch.maximum(x1.unsqueeze(2), x1.unsqueeze(1))
    iy1 = torch.maximum(y1.unsqueeze(2), y1.unsqueeze(1))
    ix2 = torch.minimum(x2.unsqueeze(2), x2.unsqueeze(1))
    iy2 = torch.minimum(y2.unsqueeze(2), y2.unsqueeze(1))
    inter = (ix2 - ix1).clamp(min=0) * (iy2 - iy1).clamp(min=0)
    a = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)
    iou = inter / (a.unsqueeze(2) + a.unsqueeze(1) - inter + 1e-6)

    t_ov = (torch.minimum(gt_times.unsqueeze(2)[..., 1], gt_times.unsqueeze(1)[..., 1]) -
            torch.maximum(gt_times.unsqueeze(2)[..., 0], gt_times.unsqueeze(1)[..., 0])).clamp(min=0)
    pair_m = cur_mask.unsqueeze(2) & cur_mask.unsqueeze(1)
    diag = ~torch.eye(C, dtype=torch.bool, device=device).unsqueeze(0)
    active = pair_m & diag & (t_ov > 0)
    if not active.any():
        return torch.zeros((), device=device, dtype=out["cx"].dtype)
    return (iou * active.float()).sum() / (active.float().sum() + 1e-6)


def d3pm_loss(out, gt_tokens, mask_target):
    """
    out: dict of (B, C, N_BINS) logits per axis
    gt_tokens: (B, C, 4)
    mask_target: (B, C, 4) bool — positions to predict
    """
    logits_stack = torch.stack([out["cx"], out["cy"], out["w"], out["h"]], dim=2)  # (B, C, 4, N_BINS)
    losses = []
    for ax in range(4):
        m = mask_target[..., ax]
        if not m.any():
            continue
        lg = logits_stack[..., ax, :][m]    # (M, N_BINS)
        tg = gt_tokens[..., ax][m]
        losses.append(F.cross_entropy(lg, tg))
    if not losses:
        return torch.zeros((), device=gt_tokens.device, dtype=torch.float32)
    return torch.stack(losses).mean()


In [8]:
# %% 셀 7.5: Sampling + IoU helpers (학습 루프 평가에서 사용)
def tokens_to_box_norm(tokens):
    cx = (tokens[..., 0].float() + 0.5) / N_BINS
    cy = (tokens[..., 1].float() + 0.5) / N_BINS
    w  = (tokens[..., 2].float() + 1) / N_BINS
    h  = (tokens[..., 3].float() + 1) / N_BINS
    return cx, cy, w, h


def iou_from_tokens(pred_tok, gt_tok):
    pcx, pcy, pw, ph = tokens_to_box_norm(pred_tok)
    gcx, gcy, gw, gh = tokens_to_box_norm(gt_tok)
    px1, py1, px2, py2 = pcx - pw/2, pcy - ph/2, pcx + pw/2, pcy + ph/2
    gx1, gy1, gx2, gy2 = gcx - gw/2, gcy - gh/2, gcx + gw/2, gcy + gh/2
    ix1 = torch.maximum(px1, gx1); iy1 = torch.maximum(py1, gy1)
    ix2 = torch.minimum(px2, gx2); iy2 = torch.minimum(py2, gy2)
    inter = (ix2 - ix1).clamp(min=0) * (iy2 - iy1).clamp(min=0)
    ap = (px2 - px1).clamp(min=0) * (py2 - py1).clamp(min=0)
    ag = (gx2 - gx1).clamp(min=0) * (gy2 - gy1).clamp(min=0)
    return inter / (ap + ag - inter + 1e-6)


def offscreen_count(pred_tok, mask):
    cx, cy, w, h = tokens_to_box_norm(pred_tok)
    out = ((cx - w/2 < 0) | (cx + w/2 > 1) | (cy - h/2 < 0) | (cy + h/2 > 1)).float()
    return (out * mask.float()).sum().item()


def pairwise_overlap_sum(pred_tok, mask):
    B, C, _ = pred_tok.shape
    cx, cy, w, h = tokens_to_box_norm(pred_tok)
    x1, x2 = cx - w/2, cx + w/2
    y1, y2 = cy - h/2, cy + h/2
    ix1 = torch.maximum(x1.unsqueeze(2), x1.unsqueeze(1))
    iy1 = torch.maximum(y1.unsqueeze(2), y1.unsqueeze(1))
    ix2 = torch.minimum(x2.unsqueeze(2), x2.unsqueeze(1))
    iy2 = torch.minimum(y2.unsqueeze(2), y2.unsqueeze(1))
    inter = (ix2 - ix1).clamp(min=0) * (iy2 - iy1).clamp(min=0)
    a = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)
    union = a.unsqueeze(2) + a.unsqueeze(1) - inter + 1e-6
    iou = inter / union
    pair_mask = mask.unsqueeze(2) & mask.unsqueeze(1)
    diag = ~torch.eye(C, dtype=torch.bool, device=mask.device).unsqueeze(0)
    return (iou * (pair_mask & diag).float()).sum(dim=(1, 2))


def maskgit_sample(model, batch, T=T_INFER, role_log_p_x=None, role_log_p_y=None,
                   role_lambda=ROLE_PRIOR_LAMBDA, temperature=1.0, gumbel_noise=True):
    model.eval()
    B, C = batch["cur_mask"].shape
    device = batch["channel_id"].device
    cur_mask = batch["cur_mask"]
    tokens = torch.full((B, C, 4), MASK_ID, dtype=torch.long, device=device)
    total_targets = (cur_mask.unsqueeze(-1).expand(-1, -1, 4)).clone()
    n_total = total_targets.sum(dim=(1, 2))

    for t_step in range(T):
        with torch.no_grad():
            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                out = model(batch, tokens)
            logits = {ax: out[ax].float() for ax in ["cx", "cy", "w", "h"]}

        if role_log_p_x is not None and role_lambda > 0:
            rp_x = role_log_p_x[batch["channel_id"]].unsqueeze(1)
            rp_y = role_log_p_y[batch["channel_id"]].unsqueeze(1)
            logits["cx"] = logits["cx"] / temperature + role_lambda * rp_x
            logits["cy"] = logits["cy"] / temperature + role_lambda * rp_y
        else:
            for ax in logits:
                logits[ax] = logits[ax] / temperature

        is_masked = (tokens == MASK_ID) & total_targets
        if not is_masked.any():
            break

        log_p_stack = torch.stack([F.log_softmax(logits["cx"], -1),
                                    F.log_softmax(logits["cy"], -1),
                                    F.log_softmax(logits["w"],  -1),
                                    F.log_softmax(logits["h"],  -1)], dim=2)
        if gumbel_noise:
            g = -torch.log(-torch.log(torch.rand_like(log_p_stack) + 1e-9) + 1e-9)
            sample_log = log_p_stack + g
        else:
            sample_log = log_p_stack
        sampled = sample_log.argmax(-1)
        confidence = log_p_stack.gather(-1, sampled.unsqueeze(-1)).squeeze(-1)
        confidence = confidence.masked_fill(~is_masked, float("-inf"))

        # ── 벡터화 top-k commit ──
        ratio = math.cos((t_step + 1) / T * math.pi / 2)
        n_keep_masked = (ratio * n_total.float()).long()                    # (B,)
        n_is_masked = is_masked.view(B, -1).sum(dim=1)                      # (B,)
        n_unmask = (n_is_masked - n_keep_masked).clamp(min=0)               # (B,)

        flat_conf = confidence.view(B, -1)                                  # (B, C*4)
        sorted_conf, sorted_idx = flat_conf.sort(dim=1, descending=True)
        L_flat = flat_conf.shape[1]
        rank = torch.arange(L_flat, device=flat_conf.device).unsqueeze(0)   # (1, L_flat)
        commit_mask_b = rank < n_unmask.unsqueeze(1)                        # (B, L_flat)

        if commit_mask_b.any():
            b_idx, k_idx = torch.where(commit_mask_b)
            flat_pos = sorted_idx[b_idx, k_idx]
            ci_all = flat_pos // 4
            ai_all = flat_pos % 4
            tokens[b_idx, ci_all, ai_all] = sampled[b_idx, ci_all, ai_all]

    leftover = (tokens == MASK_ID) & total_targets
    if leftover.any():
        with torch.no_grad():
            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                out = model(batch, tokens)
            for ax_idx, ax in enumerate(["cx", "cy", "w", "h"]):
                m = leftover[..., ax_idx]
                if m.any():
                    pred = out[ax].float().argmax(-1)
                    tokens[..., ax_idx][m] = pred[m]
    return tokens


In [ ]:
# %% 셀 8: 학습 루프
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, fused=True)
scheduler = CosineAnnealingLR(optimizer, T_max=max(1, EPOCHS * len(train_dl)))


def to_device(batch, device):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}


log_path = os.path.join(SAVE_DIR, "train_log.txt")
best_score = -float("inf")
log_lines = []


def per_epoch_eval(model, dl, n_samples=N_RERANK_SAMPLES, T_sample=T_INFER):
    """매 epoch 종합 eval — CE/accuracy/distance per-axis + sampling-based IoU/overlap."""
    model.eval()
    # 누적 (per-axis)
    ce_a = {ax: [] for ax in ["cx", "cy", "w", "h"]}
    acc_a = {ax: [] for ax in ["cx", "cy", "w", "h"]}
    dist_a = {ax: [] for ax in ["cx", "cy", "w", "h"]}
    soft_ov_list = []
    n_total_inst = 0

    # ── 1) Full-mask CE pass (전체 eval set) ──
    with torch.no_grad():
        for batch in tqdm(dl, desc="eval-ce", leave=False):
            batch = to_device(batch, DEVICE)
            cm = batch["cur_mask"]
            if not cm.any():
                continue
            noisy = batch["cur_gt_tokens"].clone()
            mask_full = cm.unsqueeze(-1).expand(-1, -1, 4)
            noisy[mask_full] = MASK_ID
            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                out = model(batch, noisy)
            # soft overlap (differentiable expected-bbox 기반 — 학습 신호와 동일)
            soft_ov = differentiable_overlap_loss(out, batch["cur_gt_tokens"], mask_full,
                                                    batch["gt_times"], cm)
            soft_ov_list.append(soft_ov.item())
            for ax_idx, ax in enumerate(["cx", "cy", "w", "h"]):
                logits = out[ax].float()
                gt = batch["cur_gt_tokens"][..., ax_idx]
                ce = F.cross_entropy(logits[cm], gt[cm], reduction="mean")
                pred = logits.argmax(-1)
                acc = (pred[cm] == gt[cm]).float().mean()
                dist = (pred[cm].float() - gt[cm].float()).abs().mean()
                ce_a[ax].append(ce.item())
                acc_a[ax].append(acc.item())
                dist_a[ax].append(dist.item())
            n_total_inst += cm.sum().item()

    # ── 2) Sampling eval (eval set 전체) ──
    sel_ious, best_ious = [], []
    enter_ious, persist_ious = [], []
    enter_best, persist_best = [], []
    ov_sums, off_rates = [], []
    with torch.no_grad():
        for batch in tqdm(dl, desc="eval-sample", leave=False):
            batch = to_device(batch, DEVICE)
            cm = batch["cur_mask"]
            if not cm.any():
                continue
            samples = []
            for _ in range(n_samples):
                tok = maskgit_sample(model, batch, T=T_sample,
                                      role_log_p_x=None, role_log_p_y=None,
                                      role_lambda=0.0)
                samples.append(tok)
            samples = torch.stack(samples, dim=0)
            N, B, C, _ = samples.shape

            # rerank by overlap
            scores = -torch.stack([pairwise_overlap_sum(samples[n], cm) for n in range(N)])
            sel_idx = scores.argmax(dim=0)
            sel_tok = samples[sel_idx, torch.arange(B, device=DEVICE)]

            iou_per = iou_from_tokens(sel_tok, batch["cur_gt_tokens"])    # (B, C)
            ious_all = torch.stack([iou_from_tokens(samples[n], batch["cur_gt_tokens"]) for n in range(N)])
            best_iou_per = ious_all.max(dim=0).values

            is_enter = batch["cur_scalars"][..., 9] > 0.5    # idx 9 = is_entering

            sel_ious.append(iou_per[cm].mean().item())
            best_ious.append(best_iou_per[cm].mean().item())
            if (is_enter & cm).any():
                enter_ious.append(iou_per[is_enter & cm].mean().item())
                enter_best.append(best_iou_per[is_enter & cm].mean().item())
            if ((~is_enter) & cm).any():
                persist_ious.append(iou_per[(~is_enter) & cm].mean().item())
                persist_best.append(best_iou_per[(~is_enter) & cm].mean().item())
            ov_sums.append(pairwise_overlap_sum(sel_tok, cm).sum().item() / max(B, 1))
            off_rates.append(offscreen_count(sel_tok, cm) / max(cm.sum().item(), 1))

    def mn(xs): return float(np.mean(xs)) if xs else 0.0
    res = {
        "ce_cx": mn(ce_a["cx"]), "ce_cy": mn(ce_a["cy"]),
        "ce_w":  mn(ce_a["w"]),  "ce_h":  mn(ce_a["h"]),
        "acc_cx": mn(acc_a["cx"]), "acc_cy": mn(acc_a["cy"]),
        "acc_w":  mn(acc_a["w"]),  "acc_h":  mn(acc_a["h"]),
        "dist_cx": mn(dist_a["cx"]), "dist_cy": mn(dist_a["cy"]),
        "dist_w":  mn(dist_a["w"]),  "dist_h":  mn(dist_a["h"]),
        "sel_iou": mn(sel_ious), "best_iou": mn(best_ious),
        "enter_iou": mn(enter_ious), "persist_iou": mn(persist_ious),
        "enter_best": mn(enter_best), "persist_best": mn(persist_best),
        "soft_ov": mn(soft_ov_list), "hard_ov": mn(ov_sums), "off_rate": mn(off_rates),
        "n_inst": n_total_inst,
    }
    res["ce_total"] = (res["ce_cx"] + res["ce_cy"] + res["ce_w"] + res["ce_h"]) / 4
    res["gap"] = res["best_iou"] - res["sel_iou"]
    return res


for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    tr_losses = []
    pbar = tqdm(train_dl, desc=f"E{epoch:02d} train")
    for batch in pbar:
        batch = to_device(batch, DEVICE)
        noisy, mask_t = corrupt_tokens(batch["cur_gt_tokens"], batch["cur_mask"])
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            out = model(batch, noisy)
            nll = d3pm_loss(out, batch["cur_gt_tokens"], mask_t)
            ov = differentiable_overlap_loss(out, batch["cur_gt_tokens"], mask_t,
                                               batch["gt_times"], batch["cur_mask"])
            loss = nll + LAMBDA_OVERLAP * ov
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        tr_losses.append(loss.item())
        pbar.set_postfix({"loss": f"{loss.item():.3f}", "nll": f"{nll.item():.3f}", "ov": f"{ov.item():.3f}"})

    tr = float(np.mean(tr_losses))
    m = per_epoch_eval(model, eval_dl)
    score = m["sel_iou"] - 0.5 * m["hard_ov"] - 0.5 * m["off_rate"]
    m["score"] = score

    line1 = (f"E{epoch:03d}  tr={tr:.4f}  ce={m['ce_total']:.4f}  "
             f"(cx={m['ce_cx']:.3f} cy={m['ce_cy']:.3f} w={m['ce_w']:.3f} h={m['ce_h']:.3f})")
    line2 = (f"        ACC  cx={m['acc_cx']:.3f} cy={m['acc_cy']:.3f} "
             f"w={m['acc_w']:.3f} h={m['acc_h']:.3f}  |  "
             f"DIST cx={m['dist_cx']:.2f} cy={m['dist_cy']:.2f} "
             f"w={m['dist_w']:.2f} h={m['dist_h']:.2f}")
    line3 = (f"        IoU  sel={m['sel_iou']:.4f} best={m['best_iou']:.4f} gap={m['gap']:.4f}  "
             f"enter(sel/best)={m['enter_iou']:.3f}/{m['enter_best']:.3f}  "
             f"persist={m['persist_iou']:.3f}/{m['persist_best']:.3f}")
    line4 = (f"        CONST soft_ov={m['soft_ov']:.4f}  hard_ov={m['hard_ov']:.4f}  off={m['off_rate']:.4f}  "
             f"score={score:.4f}  n_inst={m['n_inst']}")
    for ln in (line1, line2, line3, line4):
        print(ln); log_lines.append(ln)

    # 모든 epoch 저장
    payload = {"model": model.state_dict(), "epoch": epoch, "metrics": m}
    torch.save(payload, os.path.join(SAVE_DIR, f"epoch_{epoch:03d}.pt"))
    torch.save(payload, os.path.join(SAVE_DIR, "last.pt"))
    if score > best_score:
        best_score = score
        torch.save(payload, os.path.join(SAVE_DIR, "best.pt"))

    with open(log_path, "w") as f:
        f.write("\n".join(log_lines))


E01 train: 100%|██████████| 3971/3971 [06:24<00:00, 10.33it/s, loss=0.980, nll=0.964, ov=0.016]


E001  tr=1.5219  ce=2.6043  (cx=2.079 cy=2.669 w=4.184 h=1.486)
        ACC  cx=0.592 cy=0.390 w=0.225 h=0.585  |  DIST cx=4.18 cy=9.93 w=6.57 h=0.65
        IoU  sel=0.3883 best=0.4980 gap=0.1098  enter(sel/best)=0.184/0.316  persist=0.396/0.505
        CONST soft_ov=0.0306  hard_ov=0.4670  off=0.0195  score=0.1450  n_inst=487983


E02 train: 100%|██████████| 3971/3971 [06:05<00:00, 10.87it/s, loss=0.730, nll=0.716, ov=0.014]


E002  tr=0.7958  ce=3.1491  (cx=2.505 cy=3.060 w=5.154 h=1.877)
        ACC  cx=0.572 cy=0.379 w=0.230 h=0.593  |  DIST cx=4.15 cy=12.14 w=6.31 h=0.65
        IoU  sel=0.3644 best=0.4698 gap=0.1053  enter(sel/best)=0.173/0.306  persist=0.371/0.476
        CONST soft_ov=0.0362  hard_ov=0.5001  off=0.0281  score=0.1003  n_inst=487983


E03 train: 100%|██████████| 3971/3971 [06:23<00:00, 10.34it/s, loss=0.518, nll=0.505, ov=0.014]


...

E04 train: 100%|██████████| 3971/3971 [06:35<00:00, 10.04it/s, loss=0.555, nll=0.542, ov=0.013]


E004  tr=0.5705  ce=3.6412  (cx=2.974 cy=3.332 w=5.934 h=2.325)
        ACC  cx=0.575 cy=0.375 w=0.228 h=0.585  |  DIST cx=4.47 cy=11.80 w=6.47 h=0.65
        IoU  sel=0.3671 best=0.4702 gap=0.1031  enter(sel/best)=0.191/0.313  persist=0.374/0.476
        CONST soft_ov=0.0349  hard_ov=0.4418  off=0.0322  score=0.1301  n_inst=487983


E05 train: 100%|██████████| 3971/3971 [06:13<00:00, 10.63it/s, loss=0.465, nll=0.453, ov=0.012]


E005  tr=0.5178  ce=3.8917  (cx=3.196 cy=3.495 w=6.323 h=2.551)
        ACC  cx=0.577 cy=0.371 w=0.228 h=0.574  |  DIST cx=4.40 cy=11.79 w=6.51 h=0.67
        IoU  sel=0.3639 best=0.4616 gap=0.0978  enter(sel/best)=0.204/0.307  persist=0.370/0.467
        CONST soft_ov=0.0406  hard_ov=0.5096  off=0.0378  score=0.0902  n_inst=487983


eval-sample:  49%|████▉     | 100/204 [01:33<02:50,  1.64s/it]

In [ ]:
# %% 셀 9: best.pt 로드 + 최종 sample+rerank eval
# (helper 함수는 셀 7.5에 정의되어 있음)
# ── best.pt 로드 + best-of-N rerank eval ──
ckpt_path = os.path.join(SAVE_DIR, "best.pt")
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(ckpt["model"])
    print(f"✅ best.pt loaded (epoch {ckpt['epoch']})")
model.eval()

role_log_p_x_d = role_log_p_x.to(DEVICE)
role_log_p_y_d = role_log_p_y.to(DEVICE)

sel_ious, best_ious, off_counts, ov_sums = [], [], [], []
with torch.no_grad():
    for batch in tqdm(eval_dl, desc="sample+rerank"):
        batch = to_device(batch, DEVICE)
        # N samples
        samples = []
        for _ in range(N_RERANK_SAMPLES):
            tok = maskgit_sample(model, batch, T=T_INFER,
                                  role_log_p_x=role_log_p_x_d,
                                  role_log_p_y=role_log_p_y_d)
            samples.append(tok)
        samples = torch.stack(samples, dim=0)   # (N, B, C, 4)
        N, B, C, _ = samples.shape

        # 각 sample 점수 (overlap penalty + offscreen)
        scores = torch.zeros(N, B, device=DEVICE)
        for n in range(N):
            ov = pairwise_overlap_sum(samples[n], batch["cur_mask"])
            scores[n] = -ov   # 낮을수록 좋음

        # selected = score argmax
        sel_idx = scores.argmax(dim=0)
        sel_tok = samples[sel_idx, torch.arange(B, device=DEVICE)]    # (B, C, 4)

        # IoU 계산
        iou_per_inst = iou_from_tokens(sel_tok, batch["cur_gt_tokens"])     # (B, C)
        m = batch["cur_mask"]
        sel_ious.append(iou_per_inst[m].mean().item())

        # best-of-N: 각 instance 별로 N 샘플 중 max IoU
        ious_all = torch.stack([iou_from_tokens(samples[n], batch["cur_gt_tokens"]) for n in range(N)])  # (N, B, C)
        best_ious.append(ious_all.max(dim=0).values[m].mean().item())

        off_counts.append(offscreen_count(sel_tok, m) / max(m.sum().item(), 1))
        ov_sums.append(pairwise_overlap_sum(sel_tok, m).sum().item() / max(B, 1))

print(f"\n📊 Sample+Rerank (N={N_RERANK_SAMPLES}, T={T_INFER})")
print(f"  selected IoU: {np.mean(sel_ious):.4f}")
print(f"  best-of-N IoU: {np.mean(best_ious):.4f}")
print(f"  gap: {np.mean(best_ious) - np.mean(sel_ious):.4f}")
print(f"  off-screen rate: {np.mean(off_counts):.4f}")
print(f"  mean pair-overlap: {np.mean(ov_sums):.4f}")
